# Hartree-Fock (HF) in the atomic limit

The HF approximatino gives a self-consistent solution of $\Sigma(\nu) \approx \frac{U}{\beta}\sum_{\nu^\prime}G(\nu^\prime)e^{i0^+\nu} = U n$ and $G(\nu) \approx [G^{-1}_0(\nu)-Un]^{-1}$.

In [1]:
using MatsubaraFunctions
using NLsolve

In [ ]:
T = 0.3    # temperature
U = 0.9    # interaction
N = 1000   # number of positive grid points

1000

Initialize Green's function container:

In [ ]:
g  = MatsubaraMesh(T, N, Fermion)       # MatsubaraMesh
G  = MeshFunction(g; data_t=ComplexF64) # MeshFunction

for i in eachindex(g)
    ν = value(value(points(g,i)))
    G[i] = 1.0 / (im * ν)
end

MeshFunction{1, ComplexF64, Tuple{Mesh{MeshPoint{MatsubaraFrequency{Fermion}}, MatsubaraDomain}}, Vector{ComplexF64}}((Mesh{MeshPoint{MatsubaraFrequency{Fermion}}, MatsubaraDomain}(Symbol("7767330213297175704"), MeshPoint{MatsubaraFrequency{Fermion}}[MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 1, MatsubaraFrequency{Fermion}(0.3, -1884.013114357799, -1000)), MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 2, MatsubaraFrequency{Fermion}(0.3, -1882.1281587656451, -999)), MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 3, MatsubaraFrequency{Fermion}(0.3, -1880.243203173491, -998)), MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 4, MatsubaraFrequency{Fermion}(0.3, -1878.3582475813373, -997)), MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 5, MatsubaraFrequency{Fermion}(0.3, -1876.4732919891835, -996)), MeshPoint{MatsubaraFrequency{Fermion}}(Symbol("7767330213297175704"), 6, Ma

The Matsubara sum of the Green's function $\frac{1}{\beta}\sum_{\nu}G(\nu)e^{i\nu 0^+}\approx \frac{1}{\beta}\sum_{\nu\in\mathrm{grid}}G(\nu) + \frac{1}{2}$ is not contained in the library anymore. We have to implement ourselves.

In [ ]:
function Matsubara_sum(f::MeshFunction)
    s = zero(eltype(G.data))
    νs = meshes(G,1)
    T  = temperature(νs)
    for i in eachindex(νs)
        s += f[i] # sum over contained values
    end
    return T*s+0.5 # add 0.5 from the infinitesimal shift
end

Matsubara_sum (generic function with 1 method)

Set up fixed-point equation for NLsolve.

In [60]:
function fixed_point!(F, n, G)

    # calculate G
    for i in eachindex(g)
        ν = value(value(points(g,i)))
        G[i] = 1.0 / (im * ν - U * n[1])
    end

    # calculate the residue
    F[1] = real(Matsubara_sum(G)) - n[1]

    return nothing
end

res = nlsolve((F,n) -> fixed_point!(F, n, G), [real(Matsubara_sum(G))], method =:anderson)

Results of Nonlinear Solver Algorithm
 * Algorithm: Anderson m=1 beta=1 aa_start=1 droptol=1.0e10
 * Starting Point: [0.5]
 * Zero: [0.2932648968195736]
 * Inf-norm of residuals: 0.000000
 * Iterations: 6
 * Convergence: true
   * |x - x'| < 0.0e+00: false
   * |f(x)| < 1.0e-08: true
 * Function Calls (f): 6
 * Jacobian Calls (df/dx): 0